### 노드 (Node)
- 노드(Node)는 실제 작업을 수행하는 핵심 단위
- **노드는 Python 함수로 구현되며, 현재 상태를 입력받아 처리하고 업데이트된 상태를 반환**

<br>


In [1]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

In [2]:
class State(TypedDict):
    counter: int

In [ ]:
def increment(state: State) -> dict:
    return {"counter": state["counter"] + 1}

In [5]:
graph = StateGraph(State)
graph.add_node("increment", increment)
graph.add_edge(START, "increment")
graph.add_edge("increment", END)

In [6]:
app = graph.compile()
result = app.invoke({"counter": 0})
print(result)

{'counter': 1}


<br>

### 노드 함수 시그니처

<br>

#### 기본 시그니처 (state만)

In [ ]:
def simple_node(state: State) -> dict:
    return {"key": "value"}

<br>

#### 확장 시그니처 (state + config)
- 설정 정보가 필요할 때 `RunnableConfig`를 추가

In [8]:
from langchain_core.runnables import RunnableConfig

In [ ]:
def node_with_config(state: State, config: RunnableConfig) -> dict:
    # config에서 thread_id, tags 등 접근 가능
    thread_id = config.get("configurable", {}).get("thread_id")
    print(f"Thread ID: {thread_id}")
    return {"processed": True}

<br>

#### 전체 시그니처 (state + config + runtime)
- **스토어나 스트림 기능이 필요할 때 `Runtime` 객체를 추가**

```python
from langgraph.runtime import Runtime

def node_with_runtime(
    state: State,
    config: RunnableConfig,
    *,
    store: BaseStore,
    stream_writer: StreamWriter
) -> dict:
    # store: 장기 메모리 저장소 접근
    # stream_writer: 커스텀 스트리밍 출력
    stream_writer({"progress": "50%"})
    return {"status": "done"}
```

<br>

### 동기 vs 비동기 노드

<br>

#### 동기 노드


```python
def sync_node(state: State) -> dict:
    # 동기적으로 실행
    result = some_computation(state["data"])
    return {"result": result}
```

<br>

#### 비동기 노드
- **I/O 바운드 작업에 적합**
- 비동기 노드 사용 시 `ainvoke()` 또는 `astream()`으로 실행

```python
async def async_node(state: State) -> dict:
    # 비동기 API 호출 등
    result = await async_api_call(state["query"])
    return {"response": result}

result = await app.ainvoke({"query": "검색어"})
```

<br>

### 노드 추가 방법


<br>

#### 함수로 추가
```python
def my_node(state: State) -> dict:
    return {"key": "value"}

graph.add_node("my_node", my_node)
```

<br>

#### 이름 생략 시
- 함수 이름이 노드 이름으로 자동 지정

```python
graph.add_node(my_node)  # 노드 이름: "my_node"
```

<br>

#### `Runnable` 추가
- LangChain의 `Runnable` 객체도 노드로 추가 가능


```python
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4")
graph.add_node("llm", llm)
```

<br>




### 특수 노드

<br>

#### START 노드
- 그래프의 진입점

```python
from langgraph.graph import START

graph.add_edge(START, "first_node")
```

<br>

#### END 노드
- 그래프의 종료점

```python
from langgraph.graph import END

graph.add_edge("last_node", END)
```

<br>


#### 기본 노드 구조
- 함수 기반: Python 함수로 구현
- 상태 중심: 현재 상태를 입력으로 받아 처리
- 독립적 실행: 각 노드는 독립적으로 실행 가능
- 조합 가능: 여러 노드를 연결하여 복잡한 워크플로우 구성

In [18]:
from typing import TypedDict, Dict, Any
from langgraph.graph import StateGraph

- 상태 정의

In [19]:
class State(TypedDict):
    counter: int
    messages: list
    status: str

- 기본 노드 함수

In [20]:
def increment(state: State) -> Dict[str, Any]:
    """
    가장 간단한 형태의 노드

    Args:
        state: 현재 그래프 상태

    Returns:
        업데이트할 상태 딕셔너리
    """
    # 현재 카운터 값 가져오기
    current_counter = state["counter"]

    # 비즈니스 로직 수행
    new_counter = current_counter + 1

    # 상태 업데이트 반환
    return {
        "counter": new_counter,
        "messages": [f"카운터가 {new_counter}로 증가했습니다."],
        "status": "incremented"
    }

- 그래프 생성 및 노드 추가

In [22]:
graph = StateGraph(State)
graph.add_node("increment", increment)  # 노드 이름과 함수 매핑

<br>

#### 노드의 입출력 패턴

In [23]:
def standard_node(state: State) -> Dict[str, Any]:
    """
    표준 노드 패턴

    입력: 전체 상태
    출력: 업데이트할 필드만 포함한 딕셔너리
    """
    # 1. 상태에서 필요한 데이터 추출
    input_data = state.get("input_field", "default_value")

    # 2. 처리 로직 수행
    processed_data = process_data(input_data)

    # 3. 업데이트할 필드만 반환
    # 반환하지 않은 필드는 그대로 유지됨
    return {
        "output_field": processed_data,
        "processed": True
    }

In [24]:
def process_data(data):
    """데이터 처리 로직"""
    return f"Processed: {data}"

<br>

### 노드의 역할과 책임
- 노드의 가장 중요한 특징은 상태 중심 설계
  - 모든 노드는 공통된 상태 스키마를 통해 데이터를 주고받으며, 이를 통해 복잡한 워크플로우에서도 일관성 있는 데이터 흐름을 보장할 수 있음
  - 각 노드는 상태의 일부 또는 전체를 읽어들이고, 필요한 처리를 수행한 후, 업데이트된 상태를 반환

- 노드의 주요 책임 3가지
  - **데이터 변환 및 처리** : 원시 데이터를 구조화하거나, 복잡한 계산을 수행하거나, 외부 API와의 통신을 담당
  - **상태 관리** : 애플리케이션의 현재 상태를 읽고, 필요한 변경사항을 적용하며, 다음 단계로 전달할 정보를 준비
  - **흐름 제어** : 조건부 로직을 통해 다음에 실행될 노드를 결정하거나, 특정 조건에서 그래프의 실행을 종료시킬 수 있음

- LangGraph의 노드는 모듈성과 재사용성을 강조
  - 각 노드는 명확한 입력과 출력을 가진 독립적인 단위로 설계되어, 다른 그래프나 워크플로우에서도 쉽게 재사용할 수 있음
  - 타입 안전성을 제공하여 `TypedDict`를 활용한 상태 스키마를 통해 컴파일 시점에서 많은 오류를 방지할 수 있음

- 상태 업데이트 방식에서도 유연성을 제공
  - 기본적인 덮어쓰기 방식 외에도, 리듀서(reducer)를 사용하여 복잡한 상태 병합 로직을 구현 가능
  - 예) `Annotated[list, operator.add]`와 같은 어노테이션을 통해 리스트에 새 항목을 추가하거나, 사용자 정의 리듀서 함수를 통해 특별한 병합 규칙을 적용 가능

- 서브그래프와 `Private State` 개념을 통해 더욱 복잡한 아키텍처도 지원
  - **서브그래프는 독립적인 상태를 가지면서도 부모 그래프와 필요한 정보만을 공유할 수 있어, 대규모 멀티 에이전트 시스템이나 복잡한 워크플로우를 효과적으로 관리**

- 노드는 또한 에러 처리와 복구에도 역할
  - 각 노드에서 발생할 수 있는 예외 상황을 적절히 처리하고, 필요에 따라 재시도 로직이나 대안 경로를 제공할 수 있음
  
  $\rightarrow$ 전체 시스템의 안정성과 신뢰성 향상

- 노드는 성능 최적화의 핵심 단위
  - 각 노드의 실행 시간을 모니터링하고, 병목 지점을 식별하여 최적화할 수 있으며, 필요에 따라 병렬 처리나 캐싱 등의 기법을 적용 가능
  
  $\rightarrow$ LangGraph의 노드는 확장 가능하고 유지보수가 용이한 AI 애플리케이션 개발의 기반을 제공

<br>

#### 상태 처리 노드 함수(State Processing)

In [ ]:
class DataState(TypedDict):
    raw_data: str
    processed_data: dict
    metadata: dict


In [ ]:
def data_processor_node(state: DataState) -> Dict[str, Any]:
    """
    상태 처리 노드
    원시 데이터를 받아 구조화된 데이터로 변환
    """
    raw_data = state["raw_data"]

    # 데이터 파싱
    lines = raw_data.strip().split('\n')

    # 구조화
    processed = {
        "total_lines": len(lines),
        "content": lines,
        "first_line": lines[0] if lines else "",
        "last_line": lines[-1] if lines else ""
    }

    # 메타데이터 생성
    metadata = {
        "processed_at": datetime.now().isoformat(),
        "processor_version": "1.0.0",
        "data_size": len(raw_data)
    }

    return {
        "processed_data": processed,
        "metadata": metadata
    }

<br>

#### 비즈니스 로직 실행 노드 함수 (Business Logic Execution)

In [ ]:
class CalculationState(TypedDict):
    numbers: list[float]
    operation: str
    result: float
    history: list[dict]

In [ ]:
def calculator_node(state: CalculationState) -> Dict[str, Any]:
    """
    비즈니스 로직 실행 노드
    수학 연산을 수행하고 결과를 저장
    """
    numbers = state["numbers"]
    operation = state["operation"]

    # 연산 수행
    if operation == "sum":
        result = sum(numbers)
    elif operation == "multiply":
        result = 1
        for num in numbers:
            result *= num
    elif operation == "average":
        result = sum(numbers) / len(numbers) if numbers else 0
    else:
        raise ValueError(f"Unknown operation: {operation}")

    # 히스토리 업데이트
    history_entry = {
        "operation": operation,
        "inputs": numbers,
        "result": result,
        "timestamp": datetime.now().isoformat()
    }

    return {
        "result": result,
        "history": [history_entry]  # 리듀서가 있다면 누적됨
    }

<br>

#### 상태 업데이트 노드 함수 (State Updates)

In [ ]:
from typing import Annotated
import operator

In [ ]:
class UpdateState(TypedDict):
    counter: int
    items: Annotated[list, operator.add]  # 리듀서 적용
    flags: dict

In [ ]:
def update_node(state: UpdateState) -> Dict[str, Any]:
    """
    상태 업데이트 노드
    다양한 업데이트 패턴 시연
    """
    # 1. 단순 덮어쓰기
    new_counter = state["counter"] + 1

    # 2. 리스트에 추가 (리듀서 활용)
    new_items = ["new_item"]

    # 3. 딕셔너리 부분 업데이트
    updated_flags = state["flags"].copy()
    updated_flags["processed"] = True
    updated_flags["node_name"] = "update_node"

    return {
        "counter": new_counter,      # 덮어쓰기
        "items": new_items,          # 리듀서로 추가
        "flags": updated_flags       # 전체 교체
    }

<br>

### 노드 타입과 패턴
- 노드의 타입은 주로 실행 방식과 처리 패턴에 따라 구분되며, 이는 애플리케이션의 성능과 구조에 큰 영향
  

<br>

#### 동기 노드 (Sync Node)
- 동기 노드(Sync Node)는 가장 기본적인 형태로, 작업이 순차적으로 실행되며 완료될 때까지 다음 단계로 진행하지 않음
- 간단한 계산이나 즉시 완료되는 작업에 적합하며, 코드가 직관적이고 디버깅이 용이한 장점
- **동기 노드는 CPU 집약적인 작업이나 로컬 데이터 처리에 주로 사용**

<br>


In [25]:
def sync_node(state: State) -> Dict[str, Any]:
    """
    동기 노드 - 일반적인 노드 타입
    순차적으로 실행되며 결과를 즉시 반환
    """
    # 동기 작업 수행
    result = perform_sync_operation(state["input"])

    return {"output": result}

In [26]:
def perform_sync_operation(data):
    """동기 작업"""
    import time
    time.sleep(0.1)  # 시뮬레이션
    return f"Sync result: {data}"

<br>

#### 비동기 노드 (Async Node)
- 비동기 노드(Async Node)는 I/O 작업이나 외부 API 호출과 같이 대기 시간이 발생하는 작업에 효율적
- **`async`/`await` 패턴을 활용하여 여러 작업을 동시에 실행할 수 있으며, `asyncio.gather()`를 통해 병렬 처리를 구현**
-  네트워크 요청, 데이터베이스 쿼리, 파일 I/O 등에서 성능을 크게 향상

In [27]:
async def async_node(state: State) -> Dict[str, Any]:
    """
    비동기 노드 - I/O 작업에 효율적
    동시에 여러 작업을 처리할 수 있음
    """
    # 비동기 작업 수행
    result = await perform_async_operation(state["input"])

    # 여러 비동기 작업 동시 실행
    results = await asyncio.gather(
        fetch_data_1(),
        fetch_data_2(),
        fetch_data_3()
    )

    return {
        "output": result,
        "additional_data": results
    }

In [28]:
async def perform_async_operation(data):
    """비동기 작업"""
    await asyncio.sleep(0.1)  # 시뮬레이션
    return f"Async result: {data}"

In [29]:
async def fetch_data_1():
    await asyncio.sleep(0.05)
    return "data_1"

async def fetch_data_2():
    await asyncio.sleep(0.05)
    return "data_2"

async def fetch_data_3():
    await asyncio.sleep(0.05)
    return "data_3"

<br>

#### 조건부 노드 (Conditional Node)
- **상태나 입력에 따라 다른 처리 로직을 선택하는 유연한 패턴**

In [ ]:
def conditional_node(state: State) -> Dict[str, Any]:
    """
    조건부 노드 - 상태에 따라 다른 처리 수행
    """
    mode = state.get("mode", "default")

    if mode == "fast":
        # 빠른 처리
        result = quick_process(state)
        execution_time = "fast"
    elif mode == "thorough":
        # 정밀 처리
        result = thorough_process(state)
        execution_time = "slow"
    else:
        # 기본 처리
        result = default_process(state)
        execution_time = "normal"

    return {
        "result": result,
        "execution_time": execution_time,
        "mode_used": mode
    }

In [ ]:
def quick_process(state):
    return {"status": "quick", "data": state.get("input", "")[:10]}

def thorough_process(state):
    return {"status": "thorough", "data": analyze_deeply(state.get("input", ""))}

def default_process(state):
    return {"status": "default", "data": state.get("input", "")}

def analyze_deeply(data):
    # 심층 분석 로직
    return f"Deeply analyzed: {data}"

<br>

#### 검증 노드 (Validation Node)


In [30]:
from typing import Optional

In [ ]:
class ValidationState(TypedDict):
    input_data: Any
    validation_errors: list[str]
    is_valid: bool

In [ ]:
def validation_node(state: ValidationState) -> Dict[str, Any]:
    """
    검증 노드 - 데이터 유효성 검사
    """
    input_data = state["input_data"]
    errors = []

    # 다양한 검증 수행
    if not input_data:
        errors.append("입력 데이터가 비어있습니다.")

    if isinstance(input_data, str):
        if len(input_data) < 3:
            errors.append("문자열이 너무 짧습니다 (최소 3자).")
        if len(input_data) > 1000:
            errors.append("문자열이 너무 깁니다 (최대 1000자).")
        if not input_data.isascii():
            errors.append("ASCII 문자만 허용됩니다.")

    elif isinstance(input_data, (int, float)):
        if input_data < 0:
            errors.append("음수는 허용되지 않습니다.")
        if input_data > 1000000:
            errors.append("값이 너무 큽니다 (최대 1,000,000).")

    elif isinstance(input_data, list):
        if len(input_data) == 0:
            errors.append("리스트가 비어있습니다.")
        if len(input_data) > 100:
            errors.append("리스트 항목이 너무 많습니다 (최대 100개).")

    # 검증 결과 반환
    return {
        "validation_errors": errors,
        "is_valid": len(errors) == 0
    }

<br>


### 노드와 구성 (Configuration)
- **`RunnableConfig`는 LangChain의 핵심 구성 객체로, 노드가 실행될 때 런타임 설정을 전달받을 수 있게 함**
  - 정적으로 정의된 그래프에 동적 유연성을 부여하는 중요한 도구
  - 노드가 `RunnableConfig`를 매개변수로 받으면, 실행 시점에 다양한 구성 값들을 참조하여 처리 로직을 조정할 수 있음

- **핵심 활용 영역**
  - **모델 선택과 매개변수 조정** : 같은 노드에서 런타임에 다른 LLM 모델을 사용하거나, `temperature`, `max_tokens` 등의 매개변수를 동적으로 설정
  - **재시도 및 오류 처리 정책** : 최대 재시도 횟수, 백오프 전략, 타임아웃 설정 등을 환경이나 요청의 중요도에 따라 조정
  - **환경별 설정** : 개발, 테스트, 프로덕션 환경에 따라 다른 데이터베이스 연결, API 엔드포인트, 로깅 수준 등을 적용

- 구성 접근 패턴의 계층적 구조:
  - `config.get("configurable", {})`를 통해 사용자 정의 구성에 접근하고, 각 설정값에 대해 기본값을 제공하여 안정성을 보장
  - 예) `config.get("configurable", {}).get("model", "default")`와 같은 방식으로 안전하게 구성값을 추출할 수 있음

- 동적 처리 로직의 구현에서는 구성값에 따라 완전히 다른 알고리즘이나 처리 방식을 선택할 수 있음
  - 같은 노드가 빠른 처리 모드에서는 간단한 휴리스틱을 사용하고, 정밀 모드에서는 복잡한 AI 모델을 활용하는 것이 가능
  - 성능과 정확도 사이의 트레이드오프를 런타임에 조절할 수 있게 함
  
- 사용자별 개인화 : 사용자 ID나 권한 레벨을 구성으로 전달하여, 같은 노드에서도 사용자별로 다른 데이터 접근 권한이나 처리 방식을 적용 $\rightarrow$ 멀티테넌트 환경에서 특히 유용

- 구성의 전파와 상속에서는 부모 그래프의 구성이 서브그래프나 하위 노드로 자동 전파되며, 필요에 따라 특정 노드에서 구성을 오버라이드할 수 있음 $\rightarrow$ 계층적인 구성 관리가 가능

- 성능 최적화
  - 리소스가 제한된 환경에서는 간단한 모델을 사용하고, 고성능 환경에서는 더 복잡한 모델을 사용하도록 동적으로 조정
  - 캐싱, 배치 처리, 병렬 실행 등의 최적화 기법도 구성을 통해 제어

- 모니터링과 관찰 가능성
  - 로깅 수준, 메트릭 수집, 트레이싱 등을 환경이나 디버깅 필요에 따라 조정 $\rightarrow$ 프로덕션 환경에서의 디버깅과 성능 분석에 매우 유용

- 보안 고려사항에서는 민감한 정보가 구성에 포함될 수 있으므로, 적절한 암호화와 접근 제어가 필요
- 구성값의 유효성 검증을 통해 악의적인 입력이나 잘못된 설정으로 인한 시스템 장애를 방지해야 함

<br>

#### 런타임 구성을 받는 노드

In [31]:
from langchain_core.runnables import RunnableConfig

In [32]:
def configurable_node(state: State, config: RunnableConfig) -> Dict[str, Any]:
    """
    구성 가능한 노드
    런타임에 동작을 조정할 수 있음

    Args:
        state: 현재 상태
        config: 런타임 구성
    """
    # 구성에서 값 가져오기
    model_name = config.get("configurable", {}).get("model", "default")
    temperature = config.get("configurable", {}).get("temperature", 0.7)
    max_retries = config.get("configurable", {}).get("max_retries", 3)

    print(f"Using model: {model_name}, temperature: {temperature}")

    # 구성에 따른 처리
    result = None
    for attempt in range(max_retries):
        try:
            result = process_with_model(
                state["input"],
                model=model_name,
                temperature=temperature
            )
            break
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt == max_retries - 1:
                result = "Failed after all retries"

    return {
        "output": result,
        "config_used": {
            "model": model_name,
            "temperature": temperature,
            "retries": max_retries
        }
    }


In [33]:
def process_with_model(input_data, model, temperature):
    """모델을 사용한 처리 (시뮬레이션)"""
    return f"Processed with {model} at temperature {temperature}: {input_data}"

<br>

### 노드 구성 고급 패턴
- LangGraph에서 복잡한 애플리케이션 요구사항을 충족하기 위해 노드를 더욱 유연하고 재사용 가능하며 유지보수가 용이하도록 설계하는 방법론들을 제공

<br>

#### 1. 클래스 기반 노드
* 클래스 기반 노드는 객체지향 설계의 장점을 LangGraph에 적용한 패턴
  *  **노드가 내부 상태를 유지해야 하거나, 복잡한 로직을 캡슐화해야 하거나, 여러 메서드를 통해 기능을 분리해야 할 때 특히 유용**
- 생성자를 통해 초기 설정을 받고, `__call__` 메서드를 구현하여 호출 가능한 객체로 만들어짐
  
  $\rightarrow$ 캐싱, 통계 수집, 성능 모니터링 등의 기능을 노드 내부에서 자체적으로 관리

- **장점 : 상태 지속성과 메서드 분리**
  - 처리 횟수 추적, 캐시 관리, 설정 저장 등을 클래스의 인스턴스 변수로 관리할 수 있으며, 각기 다른 처리 방식을 별도의 메서드로 분리하여 코드의 가독성과 유지보수성을 높일 수 있음
  - 상속을 통해 기본 기능을 확장하거나 특화된 버전을 만들 수 있어 코드 재사용성이 향상

<br>

In [ ]:
class DataProcessorNode:
    """
    클래스 기반 노드
    상태를 가지고 복잡한 로직을 구현할 때 유용
    """

    def __init__(self, processor_type: str = "standard"):
        self.processor_type = processor_type
        self.processing_count = 0
        self.cache = {}

    def __call__(self, state: State) -> Dict[str, Any]:
        """
        노드 실행 메서드
        클래스 인스턴스를 호출 가능하게 만듦
        """
        input_data = state["input"]

        # 캐시 확인
        cache_key = f"{self.processor_type}:{input_data}"
        if cache_key in self.cache:
            print(f"Cache hit for {cache_key}")
            return {"output": self.cache[cache_key], "from_cache": True}

        # 처리 수행
        if self.processor_type == "standard":
            result = self._standard_process(input_data)
        elif self.processor_type == "advanced":
            result = self._advanced_process(input_data)
        else:
            result = self._default_process(input_data)

        # 통계 업데이트
        self.processing_count += 1

        # 캐시 저장
        self.cache[cache_key] = result

        return {
            "output": result,
            "processor_type": self.processor_type,
            "total_processed": self.processing_count,
            "from_cache": False
        }

    def _standard_process(self, data):
        return f"Standard processed: {data}"

    def _advanced_process(self, data):
        return f"Advanced processed: {data}"

    def _default_process(self, data):
        return f"Default processed: {data}"

# 사용 예시
processor = DataProcessorNode(processor_type="advanced")
graph.add_node("processor", processor)

<br>

#### 2. 데코레이터 패턴 노드
- 함수형 프로그래밍의 고차 함수 개념을 활용하여, 원본 노드 로직을 수정하지 않고도 새로운 기능을 추가할 수 있게 해줌
- 타이밍 측정, 에러 처리, 로깅, 캐싱, 권한 검사 등의 횡단 관심사(cross-cutting concerns)를 깔끔하게 분리할 수 있음
- **데코레이터 패턴의 핵심 가치는 관심사의 분리와 조합 가능성**
  - 예) `@timing_decorator`와 `@error_handling_decorator`를 함께 사용하여 실행 시간 측정과 에러 처리를 동시에 적용

<br>



In [34]:
from functools import wraps
import time

In [35]:
def timing_decorator(func):
    """노드 실행 시간 측정 데코레이터"""
    @wraps(func)
    def wrapper(state):
        start_time = time.time()
        result = func(state)
        execution_time = time.time() - start_time

        # 실행 시간 추가
        if isinstance(result, dict):
            result["execution_time"] = execution_time

        print(f"{func.__name__} executed in {execution_time:.3f} seconds")
        return result

    return wrapper

In [36]:
def error_handling_decorator(func):
    """에러 처리 데코레이터"""
    @wraps(func)
    def wrapper(state):
        try:
            return func(state)
        except Exception as e:
            print(f"Error in {func.__name__}: {e}")
            return {
                "error": str(e),
                "error_node": func.__name__,
                "status": "failed"
            }

    return wrapper

In [37]:
@timing_decorator
@error_handling_decorator
def decorated_node(state: State) -> Dict[str, Any]:
    """
    데코레이터가 적용된 노드
    자동으로 시간 측정과 에러 처리가 됨
    """
    # 의도적으로 느린 작업
    time.sleep(0.1)

    # 비즈니스 로직
    result = state["input"].upper() if isinstance(state["input"], str) else str(state["input"])

    return {"output": result}

<br>

#### 3. 팩토리 패턴 노드
- 런타임에 다양한 타입의 노드를 동적으로 생성하는 패턴
- 설정이나 조건에 따라 다른 동작을 하는 노드들을 일관된 인터페이스로 생성할 수 있게 함
- **팩토리 함수는 노드 타입과 매개변수를 받아 해당하는 노드 함수를 생성하고 반환 $\rightarrow$ 코드 중복을 줄이고 노드 생성 로직을 중앙화**
- **팩토리 패턴의 주요 장점은 유연성과 확장성**
  - 새로운 노드 타입을 추가할 때 기존 코드를 수정하지 않고 팩토리 함수만 확장하면 되며, 설정 파일이나 사용자 입력에 따라 동적으로 그래프 구조를 구성
  - 비슷한 기능을 하는 노드들의 공통 패턴을 추상화하여 재사용

<br>

- 팩토리 함수

In [38]:
def create_processing_node(node_type: str, **kwargs):
    """
    노드 팩토리 함수
    다양한 타입의 노드를 동적으로 생성
    """

    if node_type == "transformer":
        def transformer_node(state: State) -> Dict[str, Any]:
            transform_func = kwargs.get("transform_func", lambda x: x)
            input_field = kwargs.get("input_field", "input")
            output_field = kwargs.get("output_field", "output")

            transformed = transform_func(state[input_field])
            return {output_field: transformed}

        return transformer_node

    elif node_type == "filter":
        def filter_node(state: State) -> Dict[str, Any]:
            filter_func = kwargs.get("filter_func", lambda x: True)
            items_field = kwargs.get("items_field", "items")

            filtered = [item for item in state[items_field] if filter_func(item)]
            return {items_field: filtered}

        return filter_node

    elif node_type == "aggregator":
        def aggregator_node(state: State) -> Dict[str, Any]:
            agg_func = kwargs.get("agg_func", sum)
            values_field = kwargs.get("values_field", "values")
            result_field = kwargs.get("result_field", "result")

            aggregated = agg_func(state[values_field])
            return {result_field: aggregated}

        return aggregator_node

    else:
        raise ValueError(f"Unknown node type: {node_type}")

- 팩토리로 노드 생성

In [39]:
uppercase_node = create_processing_node(
    "transformer",
    transform_func=lambda x: x.upper(),
    input_field="text",
    output_field="uppercase_text"
)

positive_filter = create_processing_node(
    "filter",
    filter_func=lambda x: x > 0,
    items_field="numbers"
)

sum_aggregator = create_processing_node(
    "aggregator",
    agg_func=sum,
    values_field="scores",
    result_field="total_score"
)

In [ ]:
graph.add_node("uppercase", uppercase_node)
graph.add_node("filter_positive", positive_filter)
graph.add_node("calculate_sum", sum_aggregator)

<br>

<hr>